In [1]:
import os
import sys

path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(
    splitted[0] + os.sep, splitted[1], splitted[2], splitted[3]
)
src_path = os.path.join(user_independent, "GitHub", "Photoswitching")
sys.path.append(src_path)

import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fcs_p
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.prediction as pr
import src.analysis as an
import src.transitions as tr

import numpy as np
import pandas as pd
from scipy.stats import expon
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

In [3]:
fluorophores = fl.construct_fluorophores(
name="cy5_dna", distance=3, count=1, shape="square"
)
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

transitions = fluorophore_system.load_transitions(
summarize=True,
irradiance=5,
wavelength=640,
bleaching=True,
energy_transfer=True,
dstorm=True,
dstorm_parameters={'reducing_agent':'mea',
'concentration':100,
'ph':7.5},
energy_transfer_parameters={'overwrite': {'off': [1, 0.00001]}}
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.adjust_rates({8: 5e1})
transition_set.finalize()

transition_set.transition_df

transition_type abbreviation  \
Fluorophore identity                                                        
cy5_dna     0                      TransitionType.EXCITATION          EXC   
            1            TransitionType.FLUORESCENT_EMISSION          FLU   
            2         TransitionType.INTERSYSTEM_CROSSING_ST        ISCST   
            3                   TransitionType.ISOMERIZATION          ISO   
            4                      TransitionType.THERM_BISO        TBISO   
            5                     TransitionType.REDUCTION_T         REDT   
            6                     TransitionType.REDUCTION_S         REDS   
            7                     TransitionType.OXIDATION_1          OXI   
            8                TransitionType.PHOTOBLEACHING_1          BLE   
            9               TransitionType.S1_S0_TRANSITIONS      S1S0SUM   
            10              TransitionType.T1_S0_TRANSITIONS      T1S0SUM   
            11             TransitionType.CIS_S0_TRANSITIONS     CisS0SUM   

                        initial_state      final_state          rate  photon  \
Fluorophore identity                                                           
cy5_dna     0          SingleState.S0   SingleState.S1  1.453925e+07   False   
            1          SingleState.S1   SingleState.S0  1.588235e+08    True   
            2          SingleState.S1   SingleState.T1  8.300000e+05   False   
            3          SingleState.S1  SingleState.Cis  4.000000e+06   False   
            4         SingleState.Cis   SingleState.S0  5.000000e+03   False   
            5          SingleState.T1  SingleState.OFF  3.065343e+02   False   
            6          SingleState.S1  SingleState.OFF  3.065343e+03   False   
            7         SingleState.OFF   SingleState.S0  2.000000e-02   False   
            8          SingleState.T1    SingleState.B  5.000000e+01   False   
            9          SingleState.S1   SingleState.S0  4.276471e+08   False   
            10         SingleState.T1   SingleState.S0  3.115343e+05   False   
            11        SingleState.Cis   SingleState.S0  9.665504e+04   False   

                     fluorophore_ids  absorbing  
Fluorophore identity                             
cy5_dna     0                    [0]      False  
            1                    [0]      False  
            2                    [0]      False  
            3                    [0]      False  
            4                    [0]      False  
            5                    [0]      False  
            6                    [0]      False  
            7                    [0]      False  
            8                    [0]       True  
            9                    [0]      False  
            10                   [0]      False  
            11                   [0]      False

In [ ]:
sim = si.Simulation(transition_set)
sim.run(end_time=None)

WARNING for line:                     f"The smallest increment with a probability of "
 The higher limit of smallest increment with a probability of 1.00e-03 is 1.69e-12.
 This was estimated using the highest possible rate which occurs for example in state combination [1].
 Everything drawn below this number will be rounded to zero starting somewhere between 1.00e+03 - 1.00e+04. 


: 